# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is compliant with the Croissant 1.0 specification.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display the dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

Below, we use the `record_sets` property to list all record sets (`@id` and name) defined in the dataset, along with each field and column (by `@id`).

In [ ]:
# List all record sets with their fields and columns by @id
from pprint import pprint

record_sets = dataset.record_sets
if not record_sets:
    print('No record sets found in the dataset metadata.')
else:
    for rs in record_sets:
        print(f"\nRecordSet name: {rs.name}\n  @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - {field.name} (@id: {field.id})")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - {col.name} (@id: {col.id})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `record_set` and field `@id`s discovered above.

First, we'll collect all record set `@id` values into a list for extraction. You may customize this list to select specific record sets.

In [ ]:
# Extract all RecordSet @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Available RecordSet @ids:")
pprint(record_set_ids)

# Load data for each RecordSet
dataframes = {}
for recset_id in record_set_ids:
    print(f'\nLoading records for {recset_id} ...')
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded {len(df)} records: Columns are:")
        pprint(df.columns.tolist())
        print(df.head())
    else:
        print('No records found.')

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping the data.

Below, we demonstrate with one sample `RecordSet` (the first available), choosing a numeric field and a group field if available.

In [ ]:
# Select the first available DataFrame
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet with @id: {record_set_id}, shape: {df.shape}")

    # Show columns and pick a numeric field
    print("Available columns:", df.columns.tolist())
    sample_numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            sample_numeric_field = col
            break
    if sample_numeric_field is None:
        print("No numeric field available for EDA.")
    else:
        print(f"Sample numeric field identified: {sample_numeric_field}")

        # Filter records: define a threshold (e.g., mean)
        threshold = df[sample_numeric_field].mean()
        filtered_df = df[df[sample_numeric_field] > threshold].copy()
        print(f"Filtered records with {sample_numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{sample_numeric_field}_normalized"] = (
            (filtered_df[sample_numeric_field] - filtered_df[sample_numeric_field].mean()) /
            filtered_df[sample_numeric_field].std()
        )
        print(f"Normalized {sample_numeric_field} for filtered records:")
        print(filtered_df[[sample_numeric_field, f"{sample_numeric_field}_normalized"]].head())

        # Group by a candidate categorical column if available
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != sample_numeric_field:
                group_field = col
                break
        if group_field and group_field in filtered_df:
            grouped_df = filtered_df.groupby(group_field)[sample_numeric_field].mean().to_frame('mean_'+sample_numeric_field)
            print(f"Grouped data by {group_field} (mean {sample_numeric_field}):")
            print(grouped_df.head())
        else:
            print("No categorical group field identified for grouping.")
else:
    print("No tabular record sets were loaded; EDA is not possible.")

## 5. Visualization

Visualize the distribution of the numeric field and its normalized version. If a group was found, also plot groupwise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and sample_numeric_field:
    plt.figure(figsize=(6,4))
    sns.histplot(df[sample_numeric_field].dropna(), kde=True, bins=10)
    plt.title(f'Distribution of {sample_numeric_field}')
    plt.xlabel(sample_numeric_field)
    plt.show()

    if 'filtered_df' in locals() and f"{sample_numeric_field}_normalized" in filtered_df:
        plt.figure(figsize=(6,4))
        sns.histplot(filtered_df[f"{sample_numeric_field}_normalized"].dropna(), kde=True, bins=10, color='orange')
        plt.title(f'Normalized {sample_numeric_field} (Filtered)')
        plt.xlabel(f'{sample_numeric_field}_normalized')
        plt.show()

    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df.plot(kind='bar', legend=False, figsize=(8,4))
        plt.ylabel(f'Mean {sample_numeric_field}')
        plt.title(f'Mean {sample_numeric_field} by {group_field}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and explore the FAIR² dataset package.

- We identified available record sets and their fields using the Croissant schema and referenced all data entities by their `@id`.
- We loaded tabular data from the accessible record sets and demonstrated basic data exploration, normalization, grouping, and visualization.
- This workflow can be used as a template for working with any Croissant-structured FAIR dataset.

For further analysis, consider domain-specific feature engineering or integrating clinical knowledge with the supplied attributes.